In [50]:
import glob
import re
import datetime

import numpy as np
import pandas as pd

KYIV_FILE = "Kyiv.csv"
df_kyiv = pd.read_csv(KYIV_FILE)

ukraine_files = [
    f for f in glob.glob("*.csv")
    if f != KYIV_FILE and not f.endswith("_links.csv")
    and not f.endswith("_for_analysis.csv") and not f.endswith("_ML.csv")
]
print("Файли України:", ukraine_files)
df_ukraine = pd.concat([pd.read_csv(f) for f in ukraine_files], ignore_index=True)

print("Kyiv:", df_kyiv.shape)
print("Ukraine (без Києва):", df_ukraine.shape)

Файли України: ['Cherkasy.csv', 'Chernihiv.csv', 'Chernivtsi.csv', 'Dnipro.csv', 'Ivano-Frankivsk.csv', 'Kharkiv.csv', 'Kherson.csv', 'Khmelnytskyi.csv', 'Kropyvnytskyi.csv', 'Lutsk.csv', 'Lviv.csv', 'Mykolaiv.csv', 'Odesa.csv', 'Poltava.csv', 'Rivne.csv', 'Sumy.csv', 'Ternopil.csv', 'Ukraine.csv', 'Uzhhorod.csv', 'Vinnytsia.csv', 'Zaporizhzhia.csv', 'Zhytomyr.csv']
Kyiv: (11486, 39)
Ukraine (без Києва): (24114, 39)


In [51]:
# РЕФАКТОРИНГ: видалення дублікатів за group_id перенесено на самий початок
# пайплайну -- ДО будь-якої агрегації/IQR/імпутації. Раніше дедуплікація
# відбувалась лише перед one-hot кодуванням (в самому кінці), тобто всі
# попередні кроки (пороги IQR, KNN-імпутація) рахувались по "роздутих"
# дублікатами даних -- один і той самий реальний об'єкт міг враховуватись
# кілька разів і зсувати статистики.


def drop_duplicates_early(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates(subset="group_id", keep="first").copy()
    print(f"{name}: прибрано {before - len(df)} дублікатів за group_id. Лишилось: {len(df)}")
    return df


df_ukraine = drop_duplicates_early(df_ukraine, "Ukraine")
df_kyiv = drop_duplicates_early(df_kyiv, "Kyiv")

df_ukraine.to_csv("Ukraine.csv", index=False)

Ukraine: прибрано 12674 дублікатів за group_id. Лишилось: 11440
Kyiv: прибрано 620 дублікатів за group_id. Лишилось: 10866


In [52]:
df_ukraine.info()

<class 'pandas.DataFrame'>
Index: 11440 entries, 0 to 23351
Data columns (total 39 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   group_id               11440 non-null  int64  
 1   has_duplicates         11440 non-null  bool   
 2   url                    11440 non-null  str    
 3   city                   11440 non-null  str    
 4   district               11440 non-null  str    
 5   residential_complex    4795 non-null   str    
 6   lat                    11065 non-null  float64
 7   lon                    11065 non-null  float64
 8   distance_to_center_km  11065 non-null  float64
 9   poi_name               8433 non-null   object 
 10  poi_distance_m         8433 non-null   float64
 11  geo_region             11440 non-null  str    
 12  price                  11440 non-null  int64  
 13  num_of_rooms           11440 non-null  int64  
 14  area                   11440 non-null  float64
 15  living_area       

In [53]:
df_kyiv.info()

<class 'pandas.DataFrame'>
Index: 10866 entries, 0 to 11485
Data columns (total 39 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   group_id               10866 non-null  int64  
 1   has_duplicates         10866 non-null  bool   
 2   url                    10866 non-null  str    
 3   city                   10866 non-null  str    
 4   district               10866 non-null  str    
 5   residential_complex    6016 non-null   str    
 6   lat                    10858 non-null  float64
 7   lon                    10858 non-null  float64
 8   distance_to_center_km  10858 non-null  float64
 9   poi_name               2177 non-null   str    
 10  poi_distance_m         2177 non-null   float64
 11  geo_region             10866 non-null  str    
 12  price                  10866 non-null  int64  
 13  num_of_rooms           10866 non-null  int64  
 14  area                   10866 non-null  float64
 15  living_area       

In [54]:
df_ukraine.isna().sum()

group_id                    0
has_duplicates              0
url                         0
city                        0
district                    0
residential_complex      6645
lat                       375
lon                       375
distance_to_center_km     375
poi_name                 3007
poi_distance_m           3007
geo_region                  0
price                       0
num_of_rooms                0
area                        0
living_area              3542
kitchen_area               94
floor                       0
floors_in_house             0
house_type                  0
heating                     0
wall_type                   0
year_of_building         4375
ceiling_height           2666
freshly_renovated          53
bedroom_count            6680
balcony_count            7590
toilets_count            6398
kitchen_type             6809
hot_water_type           6839
has_gas                  2853
autonomy_power           8604
autonomy_heat            4867
autonomy_w

In [55]:
df_kyiv.isna().sum()

group_id                    0
has_duplicates              0
url                         0
city                        0
district                    0
residential_complex      4850
lat                         8
lon                         8
distance_to_center_km       8
poi_name                 8689
poi_distance_m           8689
geo_region                  0
price                       0
num_of_rooms                0
area                        0
living_area              4914
kitchen_area               40
floor                       0
floors_in_house             0
house_type                  0
heating                     0
wall_type                   0
year_of_building          147
ceiling_height             74
freshly_renovated         114
bedroom_count            7157
balcony_count            8037
toilets_count            7036
kitchen_type             7303
hot_water_type           7628
has_gas                  2634
autonomy_power           8273
autonomy_heat            3210
autonomy_w

In [56]:
PRICE_SQM_MIN, PRICE_SQM_MAX = 100, 10000


def clean_price_and_floors(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)
    df = df[df["price"].notna() & df["price"].between(PRICE_SQM_MIN, PRICE_SQM_MAX)]
    df = df[df["area"].notna() & (df["area"] > 0)]
    df = df[
        df["floor"].isna() | df["floors_in_house"].isna()
        | ((df["floor"] >= 1) & (df["floor"] <= df["floors_in_house"]))
    ]
    df = df[df["lat"].notna() & df["lon"].notna() & df["distance_to_center_km"].notna()]
    print(f"{name}: прибрано {before - len(df)} рядків (ціна/площа/поверховість/координати). Лишилось: {len(df)}")
    return df


df_ukraine = clean_price_and_floors(df_ukraine, "Ukraine")
df_kyiv = clean_price_and_floors(df_kyiv, "Kyiv")

Ukraine: прибрано 432 рядків (ціна/площа/поверховість/координати). Лишилось: 11008
Kyiv: прибрано 93 рядків (ціна/площа/поверховість/координати). Лишилось: 10773


In [57]:
# Пороги IQR рахуються один раз як межа "здорового глузду" для очищення
# даних (аналог бізнес-правила), а не як параметр статистичної моделі --
# тому це НЕ той тип витоку даних, який ми усуваємо нижче (той стосується
# імпутації/масштабування, які потім стають ВХОДОМ моделі). Лишаємо як є.


def room_group(df: pd.DataFrame) -> pd.Series:
    return df["num_of_rooms"].clip(upper=4).fillna(-1).astype(int).astype(str)


def remove_outliers_iqr_grouped(df: pd.DataFrame, cols: list[str], k: float = 1.5) -> pd.DataFrame:
    groups = room_group(df)
    keep_mask = pd.Series(True, index=df.index)

    for group_value in pd.unique(groups):
        idx = df.index[groups == group_value]
        group_df = df.loc[idx]
        group_mask = pd.Series(True, index=idx)
        for col in cols:
            q1, q3 = group_df[col].quantile(0.25), group_df[col].quantile(0.75)
            iqr = q3 - q1
            lower, upper = q1 - k * iqr, q3 + k * iqr
            group_mask &= group_df[col].between(lower, upper)
            print(f"  [rooms={group_value}] {col}: lower={lower:.1f}, upper={upper:.1f}")
        keep_mask.loc[idx] = group_mask

    return df[keep_mask]


before = len(df_ukraine)
df_ukraine = remove_outliers_iqr_grouped(df_ukraine, ["price", "area"])
print(f"\nПрибрано {before - len(df_ukraine)} викидів по Україні. Лишилось: {len(df_ukraine)}")

  [rooms=2] price: lower=-255.0, upper=2473.0
  [rooms=2] area: lower=10.4, upper=105.3
  [rooms=3] price: lower=-277.5, upper=2326.5
  [rooms=3] area: lower=16.2, upper=144.1
  [rooms=1] price: lower=-445.4, upper=2875.6
  [rooms=1] area: lower=11.0, upper=67.0
  [rooms=4] price: lower=-436.4, upper=2662.6
  [rooms=4] area: lower=-31.3, upper=284.4

Прибрано 717 викидів по Україні. Лишилось: 10291


In [58]:
before = len(df_kyiv)
df_kyiv = remove_outliers_iqr_grouped(df_kyiv, ["price", "area"])
print(f"\nПрибрано {before - len(df_kyiv)} викидів в Києві. Лишилось: {len(df_kyiv)}")

  [rooms=3] price: lower=-491.8, upper=4026.2
  [rooms=3] area: lower=22.0, upper=158.0
  [rooms=2] price: lower=-209.1, upper=3713.9
  [rooms=2] area: lower=15.0, upper=111.0
  [rooms=1] price: lower=49.5, upper=3637.5
  [rooms=1] area: lower=13.0, upper=69.0
  [rooms=4] price: lower=-1208.1, upper=5920.9
  [rooms=4] area: lower=9.5, upper=325.5

Прибрано 722 викидів в Києві. Лишилось: 10051


In [59]:
for df in (df_ukraine, df_kyiv):
    df["in_residential_complex"] = df["residential_complex"].notna().astype(int)
    df["has_poi"] = df["poi_name"].notna().astype(int)
    df.drop(columns=["residential_complex", "poi_name"], inplace=True)

In [60]:
# РЕФАКТОРИНГ: KNN-імпутація (year_of_building, ceiling_height, living_area,
# kitchen_area) ПРИБРАНА звідси. Раніше вона фітилась на ВСЬОМУ df (Kyiv/
# Ukraine) одразу, до train/test спліту -- значення тестових рядків (їхня
# площа/район) впливали на те, яким числом заповнювалась пропущена ознака
# сусіднього тренувального рядка, і навпаки. Це витік даних (нехай і "м'який" --
# не через сам таргет, а через сусідство в ознаках).
#
# Тепер ці колонки лишаються "сирими" з пропусками. Імпутація переїжджає в
# train-ноутбуки (train_Kyiv_ML.ipynb / train_regions_ML_flats.ipynb) і
# фітиться ЛИШЕ на train-спліті -- так само, як уже робиться для target
# encoding по місту/району.

NUMERIC_TARGETS = ["year_of_building", "ceiling_height", "living_area", "kitchen_area"]
for df, name in [(df_ukraine, "Ukraine"), (df_kyiv, "Kyiv")]:
    print(f"{name}: пропуски в {NUMERIC_TARGETS} лишаємо як є (буде імпутовано на train-спліті):")
    print(df[NUMERIC_TARGETS].isna().sum())
    print()

Ukraine: пропуски в ['year_of_building', 'ceiling_height', 'living_area', 'kitchen_area'] лишаємо як є (буде імпутовано на train-спліті):
year_of_building    3945
ceiling_height      2198
living_area         2956
kitchen_area          83
dtype: int64

Kyiv: пропуски в ['year_of_building', 'ceiling_height', 'living_area', 'kitchen_area'] лишаємо як є (буде імпутовано на train-спліті):
year_of_building     135
ceiling_height        61
living_area         4454
kitchen_area          35
dtype: int64



In [61]:
def check_area_consistency(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """NaN-safe: рядок з ще НЕ імпутованими (пропущеними) living_area/kitchen_area
    не повинен відсіюватись цією перевіркою -- лише рядки, де обидва значення
    ВІДОМІ і фізично суперечать area."""
    before = len(df)
    living_ok = df["living_area"].isna() | (df["living_area"] <= df["area"])
    kitchen_ok = df["kitchen_area"].isna() | (df["kitchen_area"] <= df["area"])
    sum_ok = (
        df["living_area"].isna() | df["kitchen_area"].isna()
        | ((df["living_area"] + df["kitchen_area"]) <= df["area"])
    )
    df = df[living_ok & kitchen_ok & sum_ok]
    print(f"{name}: прибрано {before - len(df)} рядків (непослідовні площі, де вони ВІДОМІ). Лишилось: {len(df)}")
    return df


df_ukraine = check_area_consistency(df_ukraine, "Ukraine")
df_kyiv = check_area_consistency(df_kyiv, "Kyiv")

Ukraine: прибрано 23 рядків (непослідовні площі, де вони ВІДОМІ). Лишилось: 10268
Kyiv: прибрано 2 рядків (непослідовні площі, де вони ВІДОМІ). Лишилось: 10049


In [62]:
# РЕФАКТОРИНГ: poi_distance_m тепер отримує МАРКЕРНЕ значення 99999, якщо
# точки інтересу поруч немає взагалі (has_poi == 0) -- це дає моделі чіткий,
# однозначний сигнал "дуже далеко / нерелевантно", який дерева (RandomForest,
# HistGradientBoosting) можуть використати як природний поріг для спліту.
# Раніше такі рядки заповнювались загальною медіаною -- це змішувало
# "об'єктивно немає POI" з "POI є, просто типова відстань", що є хибним
# сигналом.
#
# Якщо ж has_poi == 1, але саму відстань не вдалось визначити (рідкісна
# помилка парсингу) -- це лишається NaN і імпутується на train-спліті як
# звичайний пропуск (а не маркерним значенням, бо тут це не структурна
# відсутність POI, а брак даних).

POI_DISTANCE_SENTINEL = 99999.0

for df in (df_ukraine, df_kyiv):
    df.loc[df["has_poi"] == 0, "poi_distance_m"] = POI_DISTANCE_SENTINEL

print("Ukraine poi_distance_m NaN (has_poi=1, дистанція невідома):",
      df_ukraine.loc[df_ukraine["has_poi"] == 1, "poi_distance_m"].isna().sum())
print("Kyiv poi_distance_m NaN (has_poi=1, дистанція невідома):",
      df_kyiv.loc[df_kyiv["has_poi"] == 1, "poi_distance_m"].isna().sum())

Ukraine poi_distance_m NaN (has_poi=1, дистанція невідома): 0
Kyiv poi_distance_m NaN (has_poi=1, дистанція невідома): 0


In [63]:
# Прапорець "_unknown" -- це лише детекція пропуску (notna-перевірка), не
# статистика, що фітиться на даних, тому лишається тут без проблем. А от
# САМЕ ЗАПОВНЕННЯ модою (BOOLEAN_TRISTATE_COLS) переїжджає в train-ноутбук
# із тих самих причин, що й KNN-імпутація вище.

BOOLEAN_TRISTATE_COLS = [
    "has_gas", "autonomy_power", "autonomy_heat", "autonomy_water",
    "autonomy_net", "autonomy_lift", "freshly_renovated",
]


def flag_boolean_tristate_unknown(df: pd.DataFrame) -> pd.DataFrame:
    for col in BOOLEAN_TRISTATE_COLS:
        df[f"{col}_unknown"] = df[col].isna().astype(int)
    return df


df_ukraine = flag_boolean_tristate_unknown(df_ukraine)
df_kyiv = flag_boolean_tristate_unknown(df_kyiv)

In [64]:
# Константне заповнення "Unknown" -- це НЕ статистика, порахована з даних
# (на відміну від медіани/моди/KNN), тому не є джерелом витоку і лишається
# тут без змін.
for df in (df_ukraine, df_kyiv):
    for col in ["house_type", "wall_type", "heating", "kitchen_type", "hot_water_type"]:
        df[col] = df[col].fillna("Unknown")

df_ukraine[["house_type", "wall_type", "heating", "kitchen_type", "hot_water_type"]].isna().sum()

house_type        0
wall_type         0
heating           0
kitchen_type      0
hot_water_type    0
dtype: int64

In [65]:
# РЕФАКТОРИНГ: рахуємо прапорець "_missing" (не статистика, безпечно), але
# САМЕ заповнення значення переїжджає в train-ноутбук -- І змінюється з
# медіани на МОДУ (bedroom_count/toilets_count/balcony_count -- це малі
# цілочисельні лічильники, де "найтиповіше" значення змістовніше за медіану,
# яка на дискретному розподілі часто збігається з модою, але не завжди).

COUNT_COLS = ["bedroom_count", "toilets_count", "balcony_count"]

for df in (df_ukraine, df_kyiv):
    for col in COUNT_COLS:
        df[f"{col}_missing"] = df[col].isna().astype(int)

df_ukraine[COUNT_COLS].isna().sum()

bedroom_count    5774
toilets_count    5518
balcony_count    6621
dtype: int64

In [66]:
# --- НОВІ ознаки (feature engineering) ---
#
# РЕФАКТОРИНГ: спершу тут був building_age = CURRENT_YEAR - year_of_building
# замість year_of_building. Прибрано -- деякі об'єкти ще тільки плануються
# до будівництва (year_of_building > поточного року), і тоді "вік будинку"
# стає від'ємним і безглуздим. year_of_building сам по собі такої проблеми
# не має (просто число), тож лишаємо саме його в моделі.


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    # поверх -- перший / останній (типово інша цінова динаміка: без ліфта
    # знизу, протікання даху зверху тощо). NaN, де floor чи floors_in_house
    # невідомі -- не форсуємо 0/1 навмання.
    floor_known = df["floor"].notna() & df["floors_in_house"].notna()
    df["is_first_floor"] = np.where(floor_known, (df["floor"] == 1).astype(float), np.nan)
    df["is_last_floor"] = np.where(
        floor_known, (df["floor"] == df["floors_in_house"]).astype(float), np.nan
    )
    return df


df_ukraine = add_engineered_features(df_ukraine)
df_kyiv = add_engineered_features(df_kyiv)
df_kyiv[["year_of_building", "is_first_floor", "is_last_floor"]].describe()

,year_of_building,is_first_floor,is_last_floor
count,9914.000000,10049.000000,10049.000000
mean,1999.386020,0.053737,0.106777
std,27.080933,0.225509,0.308845
min,1858.000000,0.000000,0.000000
25%,1980.000000,0.000000,0.000000
50%,2011.000000,0.000000,0.000000
75%,2021.000000,0.000000,0.000000
max,2028.000000,1.000000,1.000000


In [67]:
# --- Ознаки з тексту оголошення (regex, ДО видалення колонки text) ---
# Прості, явні ключові слова -- не NLP/embedding, тому легко пояснити в
# роботі й перевірити вручну. Регістронезалежний пошук підрядка/патерну.

TEXT_FEATURE_PATTERNS = {
    "has_generator_text": r"генератор",
    "has_shelter_text": r"укритт|бомбосховищ|схрон",
    "has_parking_text": r"паркінг|парковк|паркомісц|машиномісц",
    "has_terrace_text": r"терас",
    "has_designer_renovation_text": r"авторськ(?:ий|ого|им)?\s*(?:ремонт|проект)|дизайнерськ(?:ий|ого|им)?\s*ремонт",
}


def add_text_features(df: pd.DataFrame) -> pd.DataFrame:
    text_lower = df["text"].fillna("").str.lower()
    for col, pattern in TEXT_FEATURE_PATTERNS.items():
        df[col] = text_lower.str.contains(pattern, regex=True, na=False).astype(int)
    return df


df_ukraine = add_text_features(df_ukraine)
df_kyiv = add_text_features(df_kyiv)
df_kyiv[list(TEXT_FEATURE_PATTERNS.keys())].mean().rename("частка згадок").to_frame()

,частка згадок
has_generator_text,0.179023
has_shelter_text,0.089462
has_parking_text,0.413773
has_terrace_text,0.047766
has_designer_renovation_text,0.116927


In [68]:
df_ukraine["price"] = df_ukraine["price"].round(0)
df_ukraine["area"] = df_ukraine["area"].round(1)
df_ukraine[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,10268.00000,10268.000000,8079.000000
mean,1110.98919,60.709875,2.740140
std,501.33642,28.805640,0.249643
min,154.00000,12.000000,2.400000
25%,738.00000,43.000000,2.550000
50%,1000.00000,54.000000,2.700000
75%,1400.00000,71.000000,2.800000
max,2875.00000,275.000000,5.000000


In [69]:
df_kyiv["price"] = df_kyiv["price"].round(0)
df_kyiv["area"] = df_kyiv["area"].round(1)
df_kyiv[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,10049.000000,10049.000000,9988.000000
mean,1815.806846,73.499692,2.776522
std,744.565242,41.239140,0.249107
min,336.000000,13.700000,2.000000
25%,1275.000000,45.700000,2.640000
50%,1635.000000,63.500000,2.700000
75%,2207.000000,88.000000,3.000000
max,5882.000000,325.000000,5.000000


In [70]:
# Дедуплікація вже відбулась на самому початку пайплайну -- тут її більше не
# повторюємо (раніше вона була саме тут, в кінці).
df_ukraine.drop(columns=["text", "group_id", "url", "has_duplicates", "lat", "lon", "without_commission", "is_exclusive"]).to_csv(
    "Ukraine_flats_for_analysis.csv", index=False
)
df_kyiv.drop(columns=["text", "group_id", "url", "has_duplicates", "lat", "lon", "without_commission", "is_exclusive"]).to_csv(
    "Kyiv_flats_for_analysis.csv", index=False
)

In [71]:
df_encoded_ukraine = df_ukraine.drop(
    columns=["url", "group_id", "geo_region", "text", "has_duplicates", "lat", "lon"]
)

categorical_cols_ukraine = ["house_type", "wall_type", "heating", "kitchen_type", "hot_water_type", "district", "city"]
df_encoded_ukraine = pd.get_dummies(df_encoded_ukraine, columns=categorical_cols_ukraine, drop_first=True)
df_encoded_ukraine.to_csv("Ukraine_flats_ML.csv", index=False)
df_encoded_ukraine.shape

(10268, 93)

In [72]:
df_encoded_kyiv = df_kyiv.drop(
    columns=["url", "group_id", "geo_region", "text", "has_duplicates", "lat", "lon"]
)

df_encoded_kyiv = df_encoded_kyiv.drop(columns=["city"])
categorical_cols_kyiv = ["house_type", "wall_type", "heating", "kitchen_type", "hot_water_type", "district"]
df_encoded_kyiv = pd.get_dummies(df_encoded_kyiv, columns=categorical_cols_kyiv, drop_first=True)
df_encoded_kyiv.to_csv("Kyiv_flats_ML.csv", index=False)
df_encoded_kyiv.shape

(10049, 82)